# Etapa 1: BiLSTM, BERTweet y XLM-R con HumAID en inglés

Entrena los tres modelos con HumAID en inglés y los evalúa en val, test (desastres no vistos), test_intra y Chile 2014 (español).

**En Kaggle:** en *Settings* elegir *Accelerator* **GPU T4 x2** o **P100** y activar *Internet*. Usar **Save Version → Save & Run All** para que los resultados queden en *Output*.

## 1. Repositorio, dependencias y datos

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/JuanTrujilloM/final-project-NN-DL.git"
BRANCH = "main"

# On Kaggle the repo is cloned; locally the notebook runs from the repo root.
if Path("/kaggle").exists():
    if not Path("final-project-NN-DL").exists():
        subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL], check=True)
    os.chdir("final-project-NN-DL")
elif Path.cwd().name == "notebooks":
    os.chdir("..")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
if not Path("data/processed/humaid_en/train.csv").exists():
    subprocess.run([sys.executable, "scripts/prepare_data.py"], check=True)
print("Directorio de trabajo:", Path.cwd())

## 2. Librerías y configuración

In [ ]:
import gc
import json
import platform
import random
import shutil
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from tokenizers import Tokenizer, models, normalizers, pre_tokenizers, trainers
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, DataCollatorWithPadding, get_linear_schedule_with_warmup

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))


def set_all_seeds(seed):
    os.environ["PL_GLOBAL_SEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_all_seeds(SEED)

In [ ]:
DATA_DIR = Path("data/processed")
RESULTS_DIR = Path("results/etapa1")
MODELS_DIR = Path("models")
TEXT_COL = "text_clean"

# None trains on all the data; a small number (e.g. 500) runs a quick test.
LIMIT = None

LABEL_NAMES = [
    "caution_and_advice",
    "displaced_people_and_evacuations",
    "infrastructure_and_utility_damage",
    "injured_or_dead_people",
    "missing_or_found_people",
    "not_humanitarian",
    "other_relevant_information",
    "requests_or_urgent_needs",
    "rescue_volunteering_or_donation_effort",
    "sympathy_and_support",
]
NUM_CLASSES = len(LABEL_NAMES)
LABEL2ID = {label: i for i, label in enumerate(LABEL_NAMES)}

EVAL_SETS = {
    "val": DATA_DIR / "humaid_en" / "val.csv",
    "test": DATA_DIR / "humaid_en" / "test.csv",
    "test_intra": DATA_DIR / "humaid_en" / "test_intra.csv",
    "chile_2014": DATA_DIR / "chile2014_es" / "test.csv",
}

## 3. Datos

In [ ]:
def load_split(path, limit=LIMIT):
    df = pd.read_csv(path)
    if limit is not None and limit < len(df):
        df = df.sample(n=limit, random_state=SEED)
    df = df.dropna(subset=[TEXT_COL]).reset_index(drop=True)
    df["label_id"] = df["label"].map(LABEL2ID).astype(int)
    return df


train_df = load_split(DATA_DIR / "humaid_en" / "train.csv")
val_df = load_split(EVAL_SETS["val"])
eval_dfs = {name: load_split(path) for name, path in EVAL_SETS.items()}

print(f"Train: {len(train_df):,} | Val: {len(val_df):,}")
for name, df in eval_dfs.items():
    print(f"  {name}: {len(df):,}")
train_df["label"].value_counts()

## 4. Métricas y reporte

In [ ]:
# F1-macro over the classes present in the true labels.
def compute_metrics(labels, preds):
    present = sorted(set(labels))
    return {
        "f1_macro": f1_score(labels, preds, labels=present, average="macro", zero_division=0),
        "accuracy": accuracy_score(labels, preds),
    }


# Saves metrics, per-class report and confusion matrix for one set.
def full_report(labels, preds, name, model_name, out_dir):
    present = sorted(set(labels))
    names = [LABEL_NAMES[i] for i in present]
    metrics = compute_metrics(labels, preds)
    metrics["n"] = len(labels)
    metrics["por_clase"] = classification_report(
        labels, preds, labels=present, target_names=names, output_dict=True, zero_division=0
    )
    (out_dir / f"metricas_{name}.json").write_text(json.dumps(metrics, indent=2, ensure_ascii=False))

    print(f"\n{'=' * 60}\n  {model_name} | {name} — F1-macro: {metrics['f1_macro']:.4f} | accuracy: {metrics['accuracy']:.4f}\n{'=' * 60}")
    print(classification_report(labels, preds, labels=present, target_names=names, digits=4, zero_division=0))

    cm = confusion_matrix(labels, preds, labels=present)
    cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    fig, axes = plt.subplots(1, 2, figsize=(22, 9))
    for ax, data, fmt, title in zip(
        axes, [cm, cm_norm], ["d", ".2f"], ["Conteos absolutos", "Proporción por clase (recall)"]
    ):
        sns.heatmap(data, annot=True, fmt=fmt, cmap="Blues", xticklabels=names, yticklabels=names, ax=ax)
        ax.set_xlabel("Predicción")
        ax.set_ylabel("Real")
        ax.set_title(f"{model_name} | {name}\n{title}")
    plt.tight_layout()
    fig.savefig(out_dir / f"confusion_{name}.png", dpi=120)
    plt.show()
    return metrics

## 5. Callbacks

In [ ]:
# Saves the best model according to a monitored metric.
class ModelCheckpoint:
    def __init__(self, filepath, monitor="val_f1_macro", mode="max", verbose=True):
        self.filepath = filepath
        self.monitor = monitor
        self.verbose = verbose
        self.best = float("-inf") if mode == "max" else float("inf")
        self.mode = mode

    def step(self, metrics, model):
        value = metrics[self.monitor]
        improved = value > self.best if self.mode == "max" else value < self.best
        if improved:
            self.best = value
            torch.save(model.state_dict(), self.filepath)
            if self.verbose:
                print(f"  ✓ Mejor modelo guardado ({self.monitor}: {value:.4f})")
        return improved


# Stops training when the metric stops improving.
class EarlyStopping:
    def __init__(self, monitor="val_f1_macro", patience=5, min_delta=1e-4, mode="max"):
        self.monitor = monitor
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.best = float("inf") if mode == "min" else float("-inf")
        self.counter = 0
        self.should_stop = False

    def step(self, metrics, model=None):
        value = metrics[self.monitor]
        improved = (value < self.best - self.min_delta if self.mode == "min"
                    else value > self.best + self.min_delta)
        if improved:
            self.best = value
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
                print(f"  ⏹ EarlyStopping activado (sin mejora en {self.patience} épocas)")
        return improved


# Lowers the learning rate when the metric plateaus.
class ReduceLROnPlateau:
    def __init__(self, optimizer, monitor="val_f1_macro", patience=3, factor=0.5, mode="max"):
        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode=mode, patience=patience, factor=factor
        )
        self.monitor = monitor

    def step(self, metrics, model=None):
        self.scheduler.step(metrics[self.monitor])

## 6. Utilidades de entrenamiento

In [ ]:
# Class weights from train: balanced, sqrt (softer) or none.
def class_weights(label_ids, mode="balanced"):
    if mode == "none":
        return None
    counts = np.bincount(np.asarray(label_ids), minlength=NUM_CLASSES).astype(float)
    weights = len(label_ids) / (NUM_CLASSES * np.maximum(counts, 1))
    if mode == "sqrt":
        weights = np.sqrt(weights)
    return torch.tensor(weights, dtype=torch.float).to(DEVICE)


# One epoch; trains if an optimizer is given, otherwise only evaluates.
def run_epoch(model, loader, criterion, forward_fn, optimizer=None, scheduler=None, scaler=None, clip_grad=1.0):
    training = optimizer is not None
    model.train() if training else model.eval()
    use_amp = scaler is not None
    total_loss, n = 0.0, 0
    all_preds, all_labels = [], []

    with torch.set_grad_enabled(training):
        for batch in loader:
            with torch.autocast(device_type=DEVICE.type, enabled=use_amp):
                logits, labels = forward_fn(model, batch)
                loss = criterion(logits.float(), labels)

            if training:
                optimizer.zero_grad()
                if use_amp:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_grad)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=clip_grad)
                    optimizer.step()
                if scheduler is not None:
                    scheduler.step()

            total_loss += loss.item() * labels.size(0)
            n += labels.size(0)
            all_preds.extend(logits.argmax(dim=1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    metrics = compute_metrics(all_labels, all_preds)
    return total_loss / n, metrics["accuracy"], metrics["f1_macro"]


@torch.no_grad()
def evaluate_clf(model, loader, forward_fn):
    model.eval()
    all_preds, all_labels = [], []
    for batch in loader:
        logits, labels = forward_fn(model, batch)
        all_preds.extend(logits.argmax(dim=1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
    return all_preds, all_labels


def new_history():
    return {"epoch": [], "phase": [], "lr": [], "train_loss": [], "val_loss": [],
            "train_acc": [], "val_acc": [], "train_f1_macro": [], "val_f1_macro": []}


def log_epoch(history, epoch, phase, lr, train, val):
    for key, value in [("epoch", epoch), ("phase", phase), ("lr", lr),
                       ("train_loss", train[0]), ("train_acc", train[1]), ("train_f1_macro", train[2]),
                       ("val_loss", val[0]), ("val_acc", val[1]), ("val_f1_macro", val[2])]:
        history[key].append(value)
    print(f"Época {epoch:02d} [{phase}] — "
          f"train loss: {train[0]:.4f}, F1-macro: {train[2]:.4f} | "
          f"val loss: {val[0]:.4f}, F1-macro: {val[2]:.4f}, acc: {val[1]:.4f} | lr: {lr:.2e}")
    return {"val_loss": val[0], "val_acc": val[1], "val_f1_macro": val[2]}


def plot_history(history, model_name, out_dir):
    pd.DataFrame(history).to_csv(out_dir / "history.csv", index=False)
    epochs = history["epoch"]
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(epochs, history["train_loss"], label="train")
    axes[0].plot(epochs, history["val_loss"], label="val", linestyle="--")
    axes[1].plot(epochs, history["train_f1_macro"], label="train")
    axes[1].plot(epochs, history["val_f1_macro"], label="val", linestyle="--")
    axes[0].set_title(f"{model_name} — Loss")
    axes[1].set_title(f"{model_name} — F1-macro")
    for ax in axes:
        ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        ax.set_xlabel("Época")
        ax.legend()
        ax.grid(alpha=0.3)
    plt.tight_layout()
    fig.savefig(out_dir / "curvas.png", dpi=120)
    plt.show()


def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total": total, "entrenables": trainable}


def hardware_info(start_time):
    info = {
        "tiempo_entrenamiento_min": round((time.time() - start_time) / 60, 2),
        "plataforma": platform.platform(),
        "torch": torch.__version__,
        "cpu_nucleos": os.cpu_count(),
        "gpu": None,
    }
    if DEVICE.type == "cuda":
        props = torch.cuda.get_device_properties(0)
        info["gpu"] = props.name
        info["gpu_memoria_gb"] = round(props.total_memory / 1e9, 1)
        info["gpu_memoria_max_usada_gb"] = round(torch.cuda.max_memory_allocated() / 1e9, 2)
    return info


# Evaluates the best model on every eval set; predictions are saved without tweet text.
def final_evaluation(model, make_loader, forward_fn, model_name, out_dir):
    results = {}
    for name, df in eval_dfs.items():
        preds, labels = evaluate_clf(model, make_loader(df), forward_fn)
        metrics = full_report(labels, preds, name, model_name, out_dir)
        results[name] = {"f1_macro": metrics["f1_macro"], "accuracy": metrics["accuracy"], "n": metrics["n"]}
        pd.DataFrame({
            "tweet_id": df["tweet_id"].astype(str),
            "event": df["event"],
            "label": [LABEL_NAMES[i] for i in labels],
            "pred": [LABEL_NAMES[i] for i in preds],
        }).to_csv(out_dir / f"predicciones_{name}.csv", index=False)
    return results


def save_summary(out_dir, **summary):
    (out_dir / "resumen.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False, default=str))

## 7. BiLSTM desde cero

In [ ]:
BILSTM = {
    "name": "bilstm_en",
    "vocab_size": 30000,
    "min_frequency": 2,
    "max_len": 80,
    "embedding_dim": 200,
    "hidden_size": 128,
    "num_layers": 2,
    "dropout": 0.3,
    "pooling": "attention",
    "batch_size": 64,
    "epochs": 30,
    "lr": 1e-3,
    "weight_decay": 0.0,
    "clip_grad": 1.0,
    "patience": 5,
    "lr_patience": 2,
    "lr_factor": 0.5,
    "class_weights": "balanced",
}

In [ ]:
PAD, UNK = "[PAD]", "[UNK]"

# Word-level tokenizer fitted on train only, so unseen words become [UNK].
word_tokenizer = Tokenizer(models.WordLevel(unk_token=UNK))
word_tokenizer.normalizer = normalizers.Lowercase()
word_tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
word_tokenizer.train_from_iterator(
    train_df[TEXT_COL].tolist(),
    trainers.WordLevelTrainer(vocab_size=BILSTM["vocab_size"], min_frequency=BILSTM["min_frequency"],
                              special_tokens=[PAD, UNK]),
)
PAD_IDX = word_tokenizer.token_to_id(PAD)
print(f"Vocabulario: {word_tokenizer.get_vocab_size():,}")


class TweetDataset(Dataset):
    def __init__(self, texts, labels, max_len):
        self.ids = [enc.ids[:max_len] or [word_tokenizer.token_to_id(UNK)]
                    for enc in word_tokenizer.encode_batch(list(texts))]
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return torch.tensor(self.ids[idx], dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)


def collate_fn(batch):
    sequences, labels = zip(*batch)
    lengths = torch.tensor([len(s) for s in sequences], dtype=torch.long)
    padded = pad_sequence(sequences, batch_first=True, padding_value=PAD_IDX)
    return padded, torch.stack(labels), lengths


def make_lstm_loader(df, shuffle=False):
    dataset = TweetDataset(df[TEXT_COL], df["label_id"], BILSTM["max_len"])
    return DataLoader(dataset, batch_size=BILSTM["batch_size"], shuffle=shuffle, collate_fn=collate_fn)


train_loader_lstm = make_lstm_loader(train_df, shuffle=True)
val_loader_lstm = make_lstm_loader(val_df)
ids, labels, lengths = next(iter(train_loader_lstm))
print("Batch:", ids.shape, labels.shape, lengths.shape)

In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes, embedding_dim, hidden_size, num_layers, dropout, pad_idx,
                 pooling="attention"):
        super().__init__()
        self.pooling = pooling
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.emb_dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        # Being bidirectional doubles the size of the LSTM output.
        if pooling == "attention":
            self.attention = nn.Linear(hidden_size * 2, 1)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, ids, lengths):
        emb = self.emb_dropout(self.embedding(ids))
        packed = pack_padded_sequence(emb, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, (h, _) = self.lstm(packed)

        if self.pooling == "last":
            rep = torch.cat([h[-2], h[-1]], dim=1)
        else:
            # Attention pooling, with padding masked out.
            out, _ = pad_packed_sequence(packed_out, batch_first=True, total_length=ids.size(1))
            mask = torch.arange(ids.size(1), device=ids.device)[None, :] < lengths[:, None]
            scores = self.attention(out).squeeze(-1).masked_fill(~mask, float("-inf"))
            weights = torch.softmax(scores, dim=1)
            rep = (weights.unsqueeze(-1) * out).sum(dim=1)

        return self.head(self.dropout(rep))


def forward_lstm(model, batch):
    ids, labels, lengths = batch
    return model(ids.to(DEVICE), lengths.to(DEVICE)), labels.to(DEVICE)

In [ ]:
set_all_seeds(SEED)
out_dir_lstm = RESULTS_DIR / BILSTM["name"]
ckpt_dir_lstm = MODELS_DIR / BILSTM["name"]
out_dir_lstm.mkdir(parents=True, exist_ok=True)
ckpt_dir_lstm.mkdir(parents=True, exist_ok=True)
word_tokenizer.save(str(ckpt_dir_lstm / "tokenizer.json"))

model_lstm = BiLSTMClassifier(
    vocab_size=word_tokenizer.get_vocab_size(), num_classes=NUM_CLASSES, embedding_dim=BILSTM["embedding_dim"],
    hidden_size=BILSTM["hidden_size"], num_layers=BILSTM["num_layers"], dropout=BILSTM["dropout"],
    pad_idx=PAD_IDX, pooling=BILSTM["pooling"],
).to(DEVICE)
print(f"Parámetros entrenables: {count_parameters(model_lstm)['entrenables']:,}")

criterion = nn.CrossEntropyLoss(weight=class_weights(train_df["label_id"], BILSTM["class_weights"]))
optimizer = torch.optim.Adam(model_lstm.parameters(), lr=BILSTM["lr"], weight_decay=BILSTM["weight_decay"])
checkpoint = ModelCheckpoint(str(ckpt_dir_lstm / "model.pt"))
reduce_lr = ReduceLROnPlateau(optimizer, patience=BILSTM["lr_patience"], factor=BILSTM["lr_factor"])
early_stop = EarlyStopping(patience=BILSTM["patience"])

history_lstm = new_history()
start_time = time.time()
if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

for epoch in range(1, BILSTM["epochs"] + 1):
    train_m = run_epoch(model_lstm, train_loader_lstm, criterion, forward_lstm, optimizer,
                        clip_grad=BILSTM["clip_grad"])
    val_m = run_epoch(model_lstm, val_loader_lstm, criterion, forward_lstm)
    metrics = log_epoch(history_lstm, epoch, "bilstm", optimizer.param_groups[0]["lr"], train_m, val_m)
    checkpoint.step(metrics, model_lstm)
    reduce_lr.step(metrics)
    early_stop.step(metrics)
    if early_stop.should_stop:
        break

resources_lstm = hardware_info(start_time)
plot_history(history_lstm, BILSTM["name"], out_dir_lstm)

In [ ]:
model_lstm.load_state_dict(torch.load(ckpt_dir_lstm / "model.pt", map_location=DEVICE))
results_lstm = final_evaluation(model_lstm, make_lstm_loader, forward_lstm, BILSTM["name"], out_dir_lstm)
save_summary(
    out_dir_lstm,
    nombre=BILSTM["name"],
    mejor_val_f1_macro=checkpoint.best,
    epocas_entrenadas=len(history_lstm["epoch"]),
    parametros=count_parameters(model_lstm),
    vocabulario=word_tokenizer.get_vocab_size(),
    recursos=resources_lstm,
    resultados=results_lstm,
    hiperparametros=BILSTM,
)
print(f"Recursos: {resources_lstm}")

## 8. Fine-tuning de Transformers (BERTweet y XLM-R)

In [ ]:
class PretrainedTweetDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.encodings = tokenizer(list(texts), truncation=True, max_length=max_len)
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: values[idx] for key, values in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item


class PretrainedClassifier(nn.Module):
    def __init__(self, model_name, num_classes, dropout=0.1, freeze_encoder=False):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)

        # BERT uses token_type_ids; RoBERTa, BERTweet and XLM-R do not.
        cfg = self.encoder.config
        self.use_token_type_ids = getattr(cfg, "type_vocab_size", 0) > 1

        if freeze_encoder:
            self.freeze_encoder()

        self.drop = nn.Dropout(dropout)
        self.classifier = nn.Linear(cfg.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = dict(input_ids=input_ids, attention_mask=attention_mask)
        if self.use_token_type_ids and token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids

        out = self.encoder(**kwargs)
        if getattr(out, "pooler_output", None) is not None:
            cls_repr = out.pooler_output
        else:
            cls_repr = out.last_hidden_state[:, 0, :]
        return self.classifier(self.drop(cls_repr))

    def freeze_encoder(self):
        for p in self.encoder.parameters():
            p.requires_grad = False

    def unfreeze_encoder(self):
        for p in self.encoder.parameters():
            p.requires_grad = True


def forward_transformer(model, batch):
    ttype = batch.get("token_type_ids")
    logits = model(
        batch["input_ids"].to(DEVICE),
        batch["attention_mask"].to(DEVICE),
        ttype.to(DEVICE) if ttype is not None else None,
    )
    return logits, batch["labels"].to(DEVICE)

In [ ]:
# Two-phase fine-tuning: frozen encoder first, then the whole model.
def finetune(hp):
    set_all_seeds(SEED)
    out_dir = RESULTS_DIR / hp["name"]
    ckpt_dir = MODELS_DIR / hp["name"]
    out_dir.mkdir(parents=True, exist_ok=True)
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    use_amp = hp["amp"] and DEVICE.type == "cuda"

    tokenizer = AutoTokenizer.from_pretrained(hp["model_name"], **hp.get("tokenizer_kwargs", {}))
    collator = DataCollatorWithPadding(tokenizer=tokenizer)
    num_workers = 2 if DEVICE.type == "cuda" else 0

    def make_loader(df, shuffle=False):
        dataset = PretrainedTweetDataset(df[TEXT_COL], df["label_id"], tokenizer, hp["max_len"])
        return DataLoader(dataset, batch_size=hp["batch_size"], shuffle=shuffle,
                          collate_fn=collator, num_workers=num_workers)

    train_loader = make_loader(train_df, shuffle=True)
    val_loader = make_loader(val_df)

    model = PretrainedClassifier(hp["model_name"], NUM_CLASSES, dropout=hp["dropout"],
                                 freeze_encoder=hp["epochs_frozen"] > 0).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights(train_df["label_id"], hp["class_weights"]))
    scaler = torch.amp.GradScaler("cuda") if use_amp else None
    checkpoint = ModelCheckpoint(str(ckpt_dir / "model.pt"))
    history = new_history()
    epoch = 0
    start_time = time.time()
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    # Phase 1: frozen encoder, only the classifier is trained.
    if hp["epochs_frozen"] > 0:
        print(f"\n[Fase 1 — encoder congelado] {count_parameters(model)['entrenables']:,} parámetros entrenables")
        opt_head = torch.optim.Adam(model.classifier.parameters(), lr=hp["lr_head"])
        for _ in range(hp["epochs_frozen"]):
            epoch += 1
            train_m = run_epoch(model, train_loader, criterion, forward_transformer, opt_head,
                                scaler=scaler, clip_grad=hp["clip_grad"])
            val_m = run_epoch(model, val_loader, criterion, forward_transformer, scaler=scaler)
            metrics = log_epoch(history, epoch, "congelado", hp["lr_head"], train_m, val_m)
            checkpoint.step(metrics, model)
        model.load_state_dict(torch.load(ckpt_dir / "model.pt", map_location=DEVICE))

    # Phase 2: whole model with a small lr and linear warmup.
    model.unfreeze_encoder()
    print(f"\n[Fase 2 — fine-tuning completo] {count_parameters(model)['entrenables']:,} parámetros entrenables")
    optimizer = torch.optim.AdamW([
        {"params": model.encoder.parameters(), "lr": hp["lr_encoder"]},
        {"params": model.classifier.parameters(), "lr": hp["lr_encoder"] * 5},
    ], weight_decay=hp["weight_decay"])
    total_steps = len(train_loader) * hp["epochs"]
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total_steps * hp["warmup_ratio"]), num_training_steps=total_steps
    )
    early_stop = EarlyStopping(patience=hp["patience"])
    for _ in range(hp["epochs"]):
        epoch += 1
        train_m = run_epoch(model, train_loader, criterion, forward_transformer, optimizer,
                            scheduler=scheduler, scaler=scaler, clip_grad=hp["clip_grad"])
        val_m = run_epoch(model, val_loader, criterion, forward_transformer, scaler=scaler)
        metrics = log_epoch(history, epoch, "fine-tuning", optimizer.param_groups[0]["lr"], train_m, val_m)
        checkpoint.step(metrics, model)
        early_stop.step(metrics)
        if early_stop.should_stop:
            break

    resources = hardware_info(start_time)
    plot_history(history, hp["name"], out_dir)

    model.load_state_dict(torch.load(ckpt_dir / "model.pt", map_location=DEVICE))
    results = final_evaluation(model, make_loader, forward_transformer, hp["name"], out_dir)
    save_summary(
        out_dir,
        nombre=hp["name"],
        modelo=hp["model_name"],
        mejor_val_f1_macro=checkpoint.best,
        epocas_entrenadas=len(history["epoch"]),
        parametros=count_parameters(model),
        recursos=resources,
        resultados=results,
        hiperparametros=hp,
    )
    print(f"Recursos: {resources}")

    # Free GPU memory before the next model.
    del model, optimizer
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return results

### 8.1 BERTweet (solo inglés)

In [ ]:
BERTWEET = {
    "name": "bertweet_en",
    "model_name": "vinai/bertweet-base",
    "tokenizer_kwargs": {"normalization": True},
    "max_len": 128,
    "dropout": 0.1,
    "batch_size": 32,
    "epochs_frozen": 1,
    "lr_head": 1e-3,
    "epochs": 4,
    "lr_encoder": 2e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "clip_grad": 1.0,
    "patience": 2,
    "class_weights": "balanced",
    "amp": True,
}
results_bertweet = finetune(BERTWEET)

### 8.2 XLM-R base (multilingüe)

In [ ]:
XLMR = {**BERTWEET, "name": "xlmr_en", "model_name": "xlm-roberta-base", "tokenizer_kwargs": {}}
results_xlmr = finetune(XLMR)

## 9. Comparación de resultados

In [ ]:
rows = []
for path in sorted(RESULTS_DIR.glob("*/resumen.json")):
    summary = json.loads(path.read_text())
    row = {
        "modelo": summary["nombre"],
        "épocas": summary["epocas_entrenadas"],
        "minutos": summary["recursos"]["tiempo_entrenamiento_min"],
        "gpu": summary["recursos"]["gpu"],
    }
    row.update({f"F1 {name}": round(r["f1_macro"], 4) for name, r in summary["resultados"].items()})
    rows.append(row)
comparison = pd.DataFrame(rows)
comparison.to_csv(RESULTS_DIR / "comparacion.csv", index=False)
comparison

In [ ]:
if Path("/kaggle").exists():
    shutil.make_archive("/kaggle/working/resultados_etapa1", "zip", RESULTS_DIR)
    print("Resultados comprimidos en /kaggle/working/resultados_etapa1.zip")